# Prueba de conexión Jupyter ↔ HDFS
Comprueba que este Jupyter puede **leer** los CSV de `/datalake/raw` y **escribir** Parquet en HDFS (WebHDFS). Ejecuta las celdas en orden.

In [1]:
import os
import pandas as pd

# Dirección de HDFS por WebHDFS (viene de las variables de entorno del contenedor)
HOST = os.environ.get('HDFS_WEBHDFS_URL', 'http://namenode:9870').replace('http://', '')
OPT = {'user': os.environ.get('HDFS_USER', 'root')}   # usuario con el que se accede a HDFS
RAW = f'webhdfs://{HOST}/datalake/raw'
CLEAN = f'webhdfs://{HOST}/datalake/clean'
print(RAW)

webhdfs://namenode:9870/datalake/raw


## 1. Leer un CSV desde HDFS

In [2]:
# Leemos solo 5 filas del CSV en bruto.
# dtype=str porque los datos están sucios (hay textos como "muchos" en columnas numéricas).
# encoding="utf-8-sig" porque los CSV llevan BOM; sin esto la primera columna sale como "\ufefffecha".
df = pd.read_csv(
    f'{RAW}/ventas_diarias_platos/01_ventas_diarias_platos.csv',
    storage_options=OPT,
    nrows=5,
    dtype=str,
    encoding='utf-8-sig',
)
df

,fecha,id_plato,precio_unidad,coste_unidad,unidades_vendidas,clientes_dia,nombre_plato_aux,franja,ciudad,moneda
0,22/02/2024,44,5.79,2.79,muchos,78,Pasta 44,cena,Madrid,EUR
1,2024-07-18,46,21.84,8.91,34,36,Ensalada 46,cena,Sevilla,EUR
2,2024-06-07,80,7.68,2.62,4,73,Paella 80,comida,Sevilla,EUR
3,2023-11-16,51,21.92,9.9,11,50,Croquetas 51,cena,Sevilla,EUR
4,2023-11-06,9,26.0,11.01,11,40,Paella 9,terraza,Valencia,EUR


## 2. Escribir un Parquet en HDFS

In [3]:
# Escribimos esas 5 filas en una carpeta de prueba dentro de /datalake/clean
df.to_parquet(f'{CLEAN}/_prueba/test.parquet', storage_options=OPT, index=False)
print('Parquet escrito')

Parquet escrito


## 3. Leerlo de vuelta y comprobar

In [4]:
df2 = pd.read_parquet(f'{CLEAN}/_prueba/test.parquet', storage_options=OPT)
print(df2.shape, '- iguales:', df.equals(df2))
df2.dtypes

(5, 10) - iguales: True


fecha                object
id_plato             object
precio_unidad        object
coste_unidad         object
unidades_vendidas    object
clientes_dia         object
nombre_plato_aux     object
franja               object
ciudad               object
moneda               object
dtype: object

## 4. Borrar el archivo de prueba

In [5]:
import fsspec
fs = fsspec.filesystem('webhdfs', host=HOST.split(':')[0], port=int(HOST.split(':')[1]), **OPT)
fs.rm(f'/datalake/clean/_prueba', recursive=True)
print('Prueba borrada:', not fs.exists('/datalake/clean/_prueba'))

Prueba borrada: True
